In [ ]:
# Imports
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from models import build_generator, build_discriminator
from data_loader import FingerprintData
from properties import calc_metrics


In [ ]:
# Configurations
config = {
    'data_path': 'data/combine',
    'label_type': 'pos',
    'latent_dim': 100,
    'fingerprint_dim': 168,
    'batch_size': 32,
    'epochs': 10,              # shorter training for demo
    'g_lr': 0.0001,
    'd_lr': 0.000001,
    'sample_interval': 10,
    'checkpoint_dir': 'models'
}

os.makedirs(config['checkpoint_dir'], exist_ok=True)
os.makedirs("outputs", exist_ok=True)


In [ ]:
# Prepare dataset
data = FingerprintData(config['data_path'], config['label_type'])
dataset = data.get_dataset(config['batch_size'])
print("Dataset loaded:", data.x.shape)

In [ ]:
# Build models
generator = build_generator(config['latent_dim'])
discriminator = build_discriminator(config['fingerprint_dim'])

g_optimizer = tf.keras.optimizers.Adam(config['g_lr'], beta_1=0.5, beta_2=0.999)
d_optimizer = tf.keras.optimizers.Adam(config['d_lr'], beta_1=0.5, beta_2=0.999)
loss_fn = tf.keras.losses.BinaryCrossentropy()


In [ ]:
# Define one training step
@tf.function
def train_step(real_data):
    batch_size = tf.shape(real_data)[0]
    noise = tf.random.normal([batch_size, config['latent_dim']])
    
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        fake_data = generator(noise, training=True)
        
        real_output = discriminator(real_data, training=True)
        fake_output = discriminator(fake_data, training=True)
        
        real_loss = loss_fn(tf.ones_like(real_output), real_output)
        fake_loss = loss_fn(tf.zeros_like(fake_output), fake_output)
        d_loss = (real_loss + fake_loss) / 2
        
        g_loss = loss_fn(tf.ones_like(fake_output), fake_output)
    
    gradients = disc_tape.gradient(d_loss, discriminator.trainable_variables)
    d_optimizer.apply_gradients(zip(gradients, discriminator.trainable_variables))
    
    gradients = gen_tape.gradient(g_loss, generator.trainable_variables)
    g_optimizer.apply_gradients(zip(gradients, generator.trainable_variables))
    
    return d_loss, g_loss


In [ ]:
# Training loop (short demo)
for epoch in range(config['epochs']):
    for real_data, _ in dataset:
        d_loss, g_loss = train_step(real_data)
    if epoch % config['sample_interval'] == 0:
        noise = tf.random.normal([1, config['latent_dim']])
        sample = generator(noise, training=False)
        plt.plot(sample.numpy().reshape(-1), label='Generated')
        plt.plot(data.x[0,:,0], label='Real')
        plt.legend()
        plt.title(f"Epoch {epoch}")
        plt.show()
    print(f"Epoch {epoch}: D Loss: {d_loss:.4f}, G Loss: {g_loss:.4f}")


In [ ]:
# Save trained model
generator.save_weights(f"{config['checkpoint_dir']}/generator_final.h5")
print("Model saved.")

In [ ]:
# Option: Load pretrained model for inference
gen = build_generator(config['latent_dim'])
gen.load_weights('./bestmodel/saved_best_model.h5')  # or use the freshly trained one
print("Pre-trained model loaded.")


In [ ]:
# Generate samples
noise = tf.random.normal([1000, config['latent_dim']])
generated = gen(noise).numpy().reshape(-1, config['fingerprint_dim'])
print("Generated samples:", generated.shape)

pd.DataFrame(np.round(generated)).head()


In [ ]:
# Evaluate generated samples
real_data = FingerprintData(config['data_path'], config['label_type']).x.reshape(-1, config['fingerprint_dim'])
metrics = calc_metrics(generated, real_data)

print("Evaluation Results:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")


In [ ]:
# Save results
output_file = "outputs/generated_samples.csv"
pd.DataFrame(np.round(generated)).to_csv(output_file, index=False)
print(f"Generated samples saved to {output_file}")
